# AOI → Sentinel-1 products: search CDSE on a webmap, pick, list the S3 paths

Draw an area of interest on a map — or paste it as WKT / GeoJSON — choose dates
and a product type, get the list of Sentinel-1 scenes covering it, tick the ones
you want and write their S3 paths to a text file, one per line. A stripped-down
Copernicus Browser inside a notebook, which stops where the download starts: the
path file is what a downloader takes, with an output folder, to fetch the
products. Nothing is downloaded here, and no credentials are needed — the CDSE
STAC catalogue is searched anonymously.

What comes out — the path file, and the AOI written next to it — is what the
next tools take as inputs: the downloader, then `polygon_to_swaths_bursts` on
the products to know which swaths and bursts cover the AOI, then the gamma0 /
coherence processing.

**Why CDSE rather than ASF.** Both catalogues list the same Sentinel-1 scenes,
but the CDSE STAC items carry the S3 key of every product in the `eodata`
bucket — exactly what the downloader wants — where ASF only hands out its own
HTTPS zip. CDSE is also the source (scenes show up there first). `asf_search`
stays the right tool for bursts as products and InSAR baselines, which is the
`asf/` folder's business, not this one's.

**GRD means the COG variant.** CDSE distributes every GRD scene twice: the
original SAFE and a Cloud-Optimised one (`..._COG.SAFE`, in the
`IW_GRDH_1S-COG` folder) — same values, same annotation XML, measurements
compressed losslessly (~30 % lighter), always online where the original of more
than a year may sit in cold storage. The STAC catalogue lists the COG only, and
this tool goes with it: SNAP reads it from version 10, the whole preprocessing
graph stays unchanged. Its product name differs from the original's (another
checksum suffix), which matters only when cross-referencing by name with
catalogues that ignore the COG, ASF or HyP3 for instance. SLC products have no
such variant.

## How to use it

**1. Install the dependencies** (all on conda-forge):

```
conda install -c conda-forge ipyleaflet ipywidgets requests geopandas shapely
```

**2. Fill the parameters cell**: where the path file goes, the form of the S3
paths the downloader expects, the initial map view and dates.

**3. Run cell 4 and work on the map.** Draw a polygon or a rectangle with the
toolbar on the left, or paste WKT / GeoJSON in the box and click *Use this AOI*.
Set the dates and criteria, *Search*. Tick products in the list — Ctrl-click for
several — and their footprints turn blue on the map while the box at the bottom
previews the lines to be written. *Write S3 paths* writes them to the path file
(overwritten each time: one file is one selection), with the AOI next to it as
`<name>_aoi.geojson` unless unticked.

**4. Cells 5 and 6 are optional**: the same search and write from plain Python —
for scripting, or if the widgets do not render — and a print of the path file.

Worth knowing:

- an IW SLC weighs ~8 GB, a GRD COG ~1.2 GB; the list shows the size of each;
- the CDSE front-end rate-limits bursts of requests (HTTP 429). The search
  retries with a back-off, so paging through a few hundred products just takes a
  little longer;
- the interface needs the Jupyter widgets front-end. It works in VS Code; if
  the map stays blank, JupyterLab is the safe bet.

## What each cell does

| # | What it does |
| --- | --- |
| 1 | This overview. |
| 2 | Imports the module sitting next to this notebook. |
| 3 | Parameters: path file, path style, map view, dates, result cap, and an AOI for the no-UI route. |
| 4 | Builds and shows the interface — map, AOI box, criteria, results list, path file. |
| 5 | The same search and write without the interface, from Python. |
| 6 | Prints the path file and says whether the AOI sits next to it. |

In [2]:
# === 2. Imports ===
from datetime import date, timedelta
from pathlib import Path
import sys

# The module sits next to this notebook
sys.path.insert(0, str(Path.cwd()))
from aoi_to_slc import build_ui, search_products, write_path_file

In [8]:
# === 3. Parameters: adapt to your setup ===

# The text file receiving the S3 paths, one product per line. A relative path
# resolves against this notebook's folder; the folder is created on the fly.
PATH_FILE = r"C:\Users\guigu\Documents\pro_asus\vigisar\data\data_raw\list.txt"

# How each line is written, to match what the downloader expects:
#     "mount"  ->  /eodata/Sentinel-1/SAR/.../<product>.SAFE
#     "s3"     ->  s3://eodata/Sentinel-1/SAR/.../<product>.SAFE
#     "key"    ->  eodata/Sentinel-1/SAR/.../<product>.SAFE
S3_PATH_STYLE = "mount"

# Initial map view (lat, lon) and zoom, and the date range the pickers open on
MAP_CENTER = (46.5, 2.5)
MAP_ZOOM = 6
END = date.today()
START = END - timedelta(days=30)

# Stop listing after that many products — narrow the dates rather than raise it
MAX_ITEMS = 300

# Used by cell 5 only, the no-UI route: inline WKT or a WKT / GeoJSON file path.
# Cell 5 prefers the AOI drawn in the interface when there is one.
AOI = "POLYGON ((-60.62689107272663 -20.61691783750756, -60.191421730006844 -20.610386275678927, -60.1835921450772 -21.03193922812448, -60.62027242536598 -21.038616712787103, -60.62689107272663 -20.61691783750756))"

In [13]:
# === 4. The interface ===
# Draw with the toolbar on the left of the map, or paste WKT / GeoJSON in the
# box and click "Use this AOI". Search, tick products (Ctrl-click for several),
# check the preview, "Write S3 paths".
#
# "Write S3 paths" overwrites PATH_FILE with the ticked products, one per line.
# With "AOI alongside" ticked, it also writes the AOI as <PATH_FILE stem>_aoi.geojson
# next to it — the next tools take products and AOI together, this keeps the
# pair in one place.
#
# Everything the interface holds is reachable from Python afterwards:
#     ui.aoi          the AOI as a shapely geometry
#     ui.results      the last search, a GeoDataFrame
#     ui.selected     the ticked rows
#     ui.s3_paths     their S3 paths, as they would be written
#     ui.written      the path files written so far
ui = build_ui(
    path_file=PATH_FILE, style=S3_PATH_STYLE, center=MAP_CENTER, zoom=MAP_ZOOM,
    start=START, end=END, max_items=MAX_ITEMS,
)
ui

In [ ]:
# === 5. The same without the interface ===
# What the buttons call, for scripting or when the widgets do not render. The
# AOI drawn in cell 4 is used when there is one, the AOI parameter otherwise.
aoi = ui.aoi if ui.aoi is not None else AOI

products = search_products(
    aoi, START, END,
    product_type="SLC",      # or "GRD" (the COG variant, see cell 1)
    mode="IW",               # "IW", "EW", "SM" or None for all
    orbit_direction=None,    # "ascending", "descending" or None
    platforms=None,          # e.g. ["S1A", "S1C"]
    max_items=MAX_ITEMS,
)
print(f"{len(products)} product(s)" + (" — truncated" if products.attrs["truncated"] else ""))
display(products.drop(columns=["geometry", "s3_key"]))

# Pick rows with pandas — one relative orbit only, say:
#     chosen = products[products.relative_orbit == 110]
# Same file as the interface: whichever runs last wins.
chosen = products.iloc[:1]
path = write_path_file(chosen, PATH_FILE, style=S3_PATH_STYLE, aoi=aoi)
print("written:", path)

In [12]:
# === 6. The path file, as the downloader will read it ===
# Shows whatever sits at PATH_FILE: a list written by cell 4 or 5, or one that
# was already there — which the next write from cell 4 or 5 will overwrite.
path = Path(PATH_FILE)
aoi_file = path.with_name(f"{path.stem}_aoi.geojson")

if path.exists():
    lines = path.read_text(encoding="utf-8").splitlines()
    print(f"{len(lines)} product(s) in {path.resolve()}")
    for line in lines:
        print("   ", line)
    if aoi_file.exists():
        print("AOI next to it:", aoi_file.name)
    else:
        print(f"no {aoi_file.name} next to it: cell 4 writes one when 'AOI alongside' "
              "is ticked, cell 5 always does")
else:
    print(f"{path} does not exist yet — write it from cell 4 or 5")

2 product(s) in C:\Users\guigu\Documents\pro_asus\vigisar\data\data_raw\list.txt
    /eodata/Sentinel-1/SAR/IW_GRDH_1S/2026/07/17/S1C_IW_GRDH_1SDV_20260717T061555_20260717T061620_008579_010FD7_614A.SAFE
    /eodata/Sentinel-1/SAR/IW_GRDH_1S/2026/07/29/S1C_IW_GRDH_1SDV_20260729T061555_20260729T061620_008754_011597_1948.SAFE
no list_aoi.geojson next to it: cell 4 writes one when 'AOI alongside' is ticked, cell 5 always does
